# Lesson 04 — Canny: The Complete Pipeline

## Why This Lesson
Canny is the gold standard edge detector. It uses a full 5-step pipeline
to produce clean, thin, connected edges. It's 40 years old and still the default choice.

## The 5 Steps
1. Gaussian blur (noise removal)
2. Gradient magnitude and direction
3. Non-maximum suppression (thin edges to 1 pixel)
4. Double threshold (strong/weak/no edge)
5. Edge tracking by hysteresis (connect weak edges to strong ones)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Basic Canny
edges_loose = cv2.Canny(gray, 50,  150)   # low thresholds = more edges
edges_tight = cv2.Canny(gray, 100, 200)   # high thresholds = only strong edges
edges_auto  = cv2.Canny(gray, 0,   0)     # not valid — need real values

# Auto Canny: use median to pick thresholds
def auto_canny(img, sigma=0.33):
    med = np.median(img)
    lo  = int(max(0,   (1-sigma) * med))
    hi  = int(min(255, (1+sigma) * med))
    return cv2.Canny(img, lo, hi), lo, hi

blurred         = cv2.GaussianBlur(gray, (5,5), 0)
edges_ac, lo, hi = auto_canny(blurred)
print(f"Auto Canny thresholds: low={lo}, high={hi}")

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, im, t in zip(axes,
    [gray, edges_loose, edges_tight, edges_ac],
    ['Grayscale', 'Loose (50,150)', 'Tight (100,200)', f'Auto ({lo},{hi})']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('Canny: lower thresholds = more edges, higher = only strongest edges', fontsize=12)
plt.show()

# Overlay edges on original
overlay = img.copy()
overlay[edges_ac > 0] = [0, 255, 0]
plt.figure(figsize=(10,6))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title('Canny edges overlaid in green on original'); plt.axis('off'); plt.show()

## Key Takeaway
Auto Canny (threshold from image median) is the best default — no manual tuning.
Always blur before Canny. The blur sigma controls how fine the edges are.